In [1]:
from src.runtime_constants import rr
from src.server.models.dms_model import DmsModel
from src import external_api
from src.external_api import write_sheet
from src.db.mysql import DbConnection

In [2]:
#rr.set_env(rr.ENV_PROD)
prompts_dms_guid = '38506c50-b1e4-4017-84f0-a2a52fa1465b'
conn = DbConnection()

In [13]:
conn.close()

In [4]:
nordic_devs = [
'Kabam',
'Tripledot',
'Stillfront',
'Miniclip',
'King',
'Electronic Arts',
'Plarium',
'Tilting Point',
'Superplus Games',
'Exient Publishing',
'Carry 1st',
'Voodoo',
'Jam City',
'Wooga',
'My Games',
'Bandai Namco',
'AppLovin'
]

In [11]:
rows = conn.select_all("""
SELECT p.app_id, p.title, p.area, p.result
  FROM google_play_scraper.gpt_prompts p
 WHERE p.status = 1
 ORDER BY p.title, p.area
""")

values = [['id', 'title', 'area', 'result']]
for row in rows:
    values.append([row['app_id'], row['title'], row['area'], row['result']])

write_sheet("1TIu0NXPH8LFgVmlzA4-y2GdI80MbnQW_-FAvAxDCvys", "gpt4 tagging", values)

In [5]:
def get_cols_from_def(conn: DbConnection):
    sheet_data = DmsModel.get_dms_data_to_json(conn, guid=prompts_dms_guid)
    cols = []
    query_cols = []
    for area in ['Genres', 'Mechanics', 'Content']:
        for it in sheet_data[area]:
            c: str = it['Category']
            t: str = it['Tag']
            t = t.replace("-", "_").replace(" ", "_")
            open_area_category = f"{area.lower()}__{c.lower()}__gpt_open_tag"
            if open_area_category not in cols:
                cols.append(open_area_category)
                query_col = f"{open_area_category} tinytext"
                query_cols.append(query_col)
            col = f"{area.lower()}__{c.lower()}__{t.lower()}"
            if col in cols:
                print(f"Duplicate col: {col}")
                continue
            query_col = f"{col} int default 0 not null"
            cols.append(col)
            query_cols.append(query_col)

    return cols, query_cols

In [48]:
sheet_data = DmsModel.get_dms_data_to_json(conn, guid=prompts_dms_guid)
cols, query_cols = get_cols_from_def(conn)
query_cols_f = ",".join(query_cols)
q = f"""
CREATE TABLE if not exists tagged_data.apps (
    id int auto_increment,
    app_id tinytext not null,
    t timestamp default NOW() not null,
    {query_cols_f},
    constraint tagged_games_pk
        primary key (id)
)
"""

conn.query(q)

Duplicate col: genres__core__stealth


In [16]:
def_cols, def_query_cols = get_cols_from_def(conn)

#print(def_cols)

rows = conn.select_all("""
SELECT p.app_id, p.title, p.area, p.result FROM google_play_scraper.gpt_prompts p WHERE p.status = 1
""")

unknown_tags = [['id', 'title', 'area', 'category', 'tag']]

data = {}
for row in rows:
    app_id = row['app_id']
    area: str = row['area']
    r: str = row['result']
    r_arr = r.split("\n")
    #print(r_arr)
    if app_id not in data:
        cols = []
        cols_query = []
        data[app_id] = {
            'cols': cols,
            'cols_query': cols_query
        }
    else:
        cols = data[app_id]['cols']
        cols_query = data[app_id]['cols_query']

    for it in r_arr:
        s: str = it
        s = s.removeprefix("- ")
        while "(" in s and ")" in s:
            xi1 = s.index("(")
            xi2 = s.index(")")
            s = s[:xi1] + s[xi2 + 1:]
        s_arr = s.split(": ")
        if len(s_arr) != 2:
            print(f"Error parsing {app_id} - {area}: {s}")
            continue
        c = s_arr[0].strip()
        t_arr = s_arr[1].split(", ")

        for t in t_arr:
            t_orig = t.strip()
            t_orig = t_orig.removesuffix("_")
            t_orig = t_orig.removesuffix(")")
            t_orig = t_orig.strip()
            t = t_orig
            t = t.replace("-", "_").replace(" ", "_")

            t = t.lower()
            if t == "none":
                continue

            col = f"{area.lower()}__{c.lower()}__{t}"
            if col not in def_cols:
                unknown_tags.append([app_id, row['title'], area, c, t_orig])
                #    print(f"Unknown column: {col}")
                col = f"{area.lower()}__{c.lower()}__gpt_open_tag"
                col_query = f"{col} = '{t}'"
            else:
                col_query = f"{col} = 1"
            cols.append(col)
            cols_query.append(col_query)

print(unknown_tags)
write_sheet("1TIu0NXPH8LFgVmlzA4-y2GdI80MbnQW_-FAvAxDCvys", "gpt4 unknown tags", unknown_tags)


cnt = 10
if False:
    for app_id in data:
        if len(data[app_id]['cols_query']) > 0:
            print(f"Adding: {app_id}")
            q_cols = ",".join(data[app_id]['cols_query'])
            try:
                conn.query(f"""
                DELETE FROM tagged_data.apps tg WHERE tg.app_id = '{app_id}'
                """)
                conn.query(f"""
                INSERT INTO tagged_data.apps (app_id) VALUES ('{app_id}')
                """)
                q = f"UPDATE tagged_data.apps tg SET {q_cols} WHERE tg.app_id = '{app_id}'"
                conn.query(q)
            except Exception as err:
                print(str(err))
                conn.rollback()
                continue

            conn.commit()

            cnt -= 1
            #if cnt <= 0:
            #    break


Duplicate col: genres__core__stealth
Error parsing com.superplusgames.hosandroid - Content: 
Error parsing com.superplusgames.hosandroid - Content: Based on the provided store description, the applicable tags for Hills of Steel are:
Error parsing com.superplusgames.hosandroid - Content: 
Error parsing com.tapped.flipdunk - Content: 
Error parsing com.tapped.flipdunk - Content: 
[['id', 'title', 'area', 'category', 'tag'], ['com.playgendary.kickthebuddy', 'Kick the Buddy', 'Genres', 'Core', 'Destruction'], ['com.ea.game.nfs14_row', 'Need for Speed™ No Limits', 'Genres', 'Core', 'Customization'], ['com.plarium.military', 'Soldiers Inc: Mobile Warfare', 'Genres', 'Core', 'Teamwork'], ['com.playrix.gardenscapes', 'Gardenscapes', 'Genres', 'Core', 'Restore'], ['com.playrix.gardenscapes', 'Gardenscapes', 'Genres', 'Core', 'Decorate'], ['net.wooga.junes_journey_hidden_object_mystery_game', "June's Journey: Hidden Objects", 'Genres', 'Core', 'Decorating'], ['net.wooga.junes_journey_hidden_obje

In [23]:
rows = conn.select_all("""
SELECT tg.* FROM tagged_data.apps tg
""")

h = []
values = [h]
for row in rows:
    if len(h) == 0:
        for k in row:
            if k == 't':
                continue
            h.append(k)
    v = [0] * len(h)
    values.append(v)
    for k in row:
        if k == 't':
            continue
        ix = h.index(k)
        v[ix] = row[k]

len(values)
write_sheet("1TIu0NXPH8LFgVmlzA4-y2GdI80MbnQW_-FAvAxDCvys", "gpt4 tagged games", values)


{'spreadsheetId': '1TIu0NXPH8LFgVmlzA4-y2GdI80MbnQW_-FAvAxDCvys',
 'updatedRange': "'gpt4 tagged games'!A1:OE154",
 'updatedRows': 154,
 'updatedColumns': 395,
 'updatedCells': 54937}